In [1]:
import nltk

def ngrams(sentence, n):
    words = sentence.split()
    ngrams = zip(*[words[i:] for i in range(n)])
    return list(ngrams)

sentence = "안녕하세요. 만나서 진심으로 반가워요."

unigram = ngrams(sentence, 1)
bigram = ngrams(sentence, 2)
trigram = ngrams(sentence, 3)

print(unigram)
print(bigram)
print(trigram)

unigram = nltk.ngrams(sentence.split(), 1)
bigram = nltk.ngrams(sentence.split(), 2)
trigram = nltk.ngrams(sentence.split(), 3)

print(list(unigram))
print(list(bigram))
print(list(trigram))

[('안녕하세요.',), ('만나서',), ('진심으로',), ('반가워요.',)]
[('안녕하세요.', '만나서'), ('만나서', '진심으로'), ('진심으로', '반가워요.')]
[('안녕하세요.', '만나서', '진심으로'), ('만나서', '진심으로', '반가워요.')]
[('안녕하세요.',), ('만나서',), ('진심으로',), ('반가워요.',)]
[('안녕하세요.', '만나서'), ('만나서', '진심으로'), ('진심으로', '반가워요.')]
[('안녕하세요.', '만나서', '진심으로'), ('만나서', '진심으로', '반가워요.')]


In [ ]:
# tfidf 를 사용한 벡터화
from sklearn.feature_extraction.text import TfidfVectorizer

corpus = ["That movie is famous movie",
          "I like that actor",
          "I don't like that actor"]

tfidf_vectorizer = TfidfVectorizer()
tfidf_vectorizer.fit(corpus)
tfidf_matrix = tfidf_vectorizer.transform(corpus)

print(tfidf_matrix.toarray())
print(tfidf_vectorizer.vocabulary_)

[[0.         0.         0.39687454 0.39687454 0.         0.79374908
  0.2344005 ]
 [0.61980538 0.         0.         0.         0.61980538 0.
  0.48133417]
 [0.4804584  0.63174505 0.         0.         0.4804584  0.
  0.37311881]]
{'that': 6, 'movie': 5, 'is': 3, 'famous': 2, 'like': 4, 'actor': 0, 'don': 1}


In [3]:
# skipgram 모델 실습

import os
import torch.nn as nn

class VanillaSkipgram(nn.Module):
    def __init__(self, vocab_size, embedding_dim):
        super().__init__()
        self.embedding = nn.Embedding(num_embeddings = vocab_size,
                                      embedding_dim = embedding_dim)
        self.linear = nn.Linear(in_features = embedding_dim,
                                out_features = vocab_size)
    def forward(self, input_ids):
        embeddings = self.embedding(input_ids)
        output = self.linear(embeddings)
        return output

In [4]:
import pandas as pd
from Korpora import Korpora
from konlpy.tag import Okt

corpus = Korpora.load("nsmc")
corpus = pd.DataFrame(corpus.test)


    Korpora 는 다른 분들이 연구 목적으로 공유해주신 말뭉치들을
    손쉽게 다운로드, 사용할 수 있는 기능만을 제공합니다.

    말뭉치들을 공유해 주신 분들에게 감사드리며, 각 말뭉치 별 설명과 라이센스를 공유 드립니다.
    해당 말뭉치에 대해 자세히 알고 싶으신 분은 아래의 description 을 참고,
    해당 말뭉치를 연구/상용의 목적으로 이용하실 때에는 아래의 라이센스를 참고해 주시기 바랍니다.

    # Description
    Author : e9t@github
    Repository : https://github.com/e9t/nsmc
    References : www.lucypark.kr/docs/2015-pyconkr/#39

    Naver sentiment movie corpus v1.0
    This is a movie review dataset in the Korean language.
    Reviews were scraped from Naver Movies.

    The dataset construction is based on the method noted in
    [Large movie review dataset][^1] from Maas et al., 2011.

    [^1]: http://ai.stanford.edu/~amaas/data/sentiment/

    # License
    CC0 1.0 Universal (CC0 1.0) Public Domain Dedication
    Details in https://creativecommons.org/publicdomain/zero/1.0/



[nsmc] download ratings_train.txt: 14.6MB [00:00, 58.3MB/s]                            
[nsmc] download ratings_test.txt: 4.90MB [00:00, 25.2MB/s]                            


In [5]:
# tokenizer 사용한 분해?

tokenizer = Okt()
tokens = [tokenizer.morphs(review) for review in corpus.text]
print(tokens[:3])

[['굳', 'ㅋ'], ['GDNTOPCLASSINTHECLUB'], ['뭐', '야', '이', '평점', '들', '은', '....', '나쁘진', '않지만', '10', '점', '짜', '리', '는', '더', '더욱', '아니잖아']]


In [6]:
from collections import Counter

def build_vocab(corpus, n_vocab, special_tokens):
    counter = Counter()
    for tokens in corpus:
        counter.update(tokens)
    vocab = special_tokens
    for token, count in counter.most_common(n_vocab):
        vocab.append(token)
    return vocab

vocab = build_vocab(corpus = tokens,
                    n_vocab = 5000,
                    special_tokens = ["<unk>"])
token_to_id = {token: idx for idx, token in enumerate(vocab)}
id_to_token = {idx: token for idx, token in enumerate(vocab)}

print(vocab[:10])
print(len(vocab))

['<unk>', '.', '이', '영화', '의', '..', '가', '에', '...', '을']
5001


In [7]:
def get_word_pairs(tokens, window_size):
    pairs = []
    for sentence in tokens:
        sentence_length = len(sentence)
        for idx, center_word in enumerate(sentence):
            window_start = max(0, idx - window_size)
            window_end = min(sentence_length, idx + window_size + 1)
            center_word = sentence[idx]
            context_words = sentence[window_start:idx] + sentence[idx + 1 : window_end]
            for context_word in context_words:
                pairs.append([center_word, context_word])
    return pairs
word_pairs = get_word_pairs(tokens, window_size = 2)
print(word_pairs[:5])

[['굳', 'ㅋ'], ['ㅋ', '굳'], ['뭐', '야'], ['뭐', '이'], ['야', '뭐']]


In [8]:
def get_index_pairs(word_pairs, token_to_id):
    pairs = []
    unk_index = token_to_id["<unk>"]
    for word_pair in word_pairs:
        center_word, context_word = word_pair
        center_index = token_to_id.get(center_word, unk_index)
        context_index = token_to_id.get(context_word, unk_index)
        pairs.append([center_index, context_index])
    return pairs

index_pairs = get_index_pairs(word_pairs, token_to_id)
print(index_pairs[:5])
print(len(vocab))

[[595, 100], [100, 595], [77, 176], [77, 2], [176, 77]]
5001


In [9]:
import torch
from torch.utils.data import TensorDataset, DataLoader

index_pairs = torch.tensor(index_pairs)
center_indexes = index_pairs[:, 0]
context_indexes = index_pairs[:, 1]

dataset = TensorDataset(center_indexes, context_indexes)
dataloader = DataLoader(dataset, batch_size = 32, shuffle = True)

In [10]:
import torch.optim as optim

device = "cuda" if torch.cuda.is_available() else "cpu"
print(device)
word2vec = VanillaSkipgram(vocab_size = len(token_to_id),
                           embedding_dim = 128).to(device)
criterion = nn.CrossEntropyLoss().to(device)
optimizer = optim.SGD(word2vec.parameters(), lr = 0.1)

cuda


In [11]:
for epoch in range(10):
    cost = 0.0
    for input_ids, target_ids in dataloader:
        input_ids = input_ids.to(device)
        target_ids = target_ids.to(device)

        logits = word2vec(input_ids)
        loss = criterion(logits, target_ids)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        cost += loss
    cost = cost / len(dataloader)
    print(epoch + 1, cost)

1 tensor(6.1974, device='cuda:0', grad_fn=<DivBackward0>)
2 tensor(5.9825, device='cuda:0', grad_fn=<DivBackward0>)
3 tensor(5.9333, device='cuda:0', grad_fn=<DivBackward0>)
4 tensor(5.9029, device='cuda:0', grad_fn=<DivBackward0>)
5 tensor(5.8807, device='cuda:0', grad_fn=<DivBackward0>)
6 tensor(5.8628, device='cuda:0', grad_fn=<DivBackward0>)
7 tensor(5.8479, device='cuda:0', grad_fn=<DivBackward0>)
8 tensor(5.8349, device='cuda:0', grad_fn=<DivBackward0>)
9 tensor(5.8236, device='cuda:0', grad_fn=<DivBackward0>)
10 tensor(5.8131, device='cuda:0', grad_fn=<DivBackward0>)


In [12]:
token_to_embedding = dict()
embedding_matrix = word2vec.embedding.weight.detach().cpu()

for word, embedding in zip(vocab, embedding_matrix):
    token_to_embedding[word] = embedding
index = 30
token = vocab[index]
token_embedding = token_to_embedding[token]
print(token)
print(token_embedding)

연기
tensor([-0.8848, -0.8897,  0.6469, -1.3956, -1.0851, -0.3401,  0.1945,  1.0183,
        -0.1789,  0.2023,  0.2397, -0.3643, -1.0138,  0.5973,  0.0430,  0.5764,
        -0.1737, -0.3103,  0.4461,  0.0274,  0.2290, -0.9691,  0.2715,  0.0648,
        -0.6838, -1.7934,  0.8048,  0.4119,  0.6271,  0.2238,  1.5198,  0.6601,
        -0.8976,  0.7973, -0.2603, -1.6469,  0.7098,  0.9944,  0.8351,  0.5842,
         0.9730, -0.6291,  0.5417,  2.2293,  0.2795,  0.5179, -0.0943,  0.4045,
         0.1139,  0.4148, -0.1242,  1.6999, -0.7651, -0.1404,  1.1255,  1.5249,
        -0.5968, -2.0091, -0.2741, -1.9862,  0.6465,  1.3520, -1.5117,  0.8245,
         1.7718,  0.9304, -0.5738,  0.0091, -1.8655, -0.1035,  1.6635,  0.6581,
        -0.0854, -0.3921,  0.0435, -1.2616,  0.5603, -0.6802,  0.7945, -0.0640,
         0.5241, -0.2214, -1.4607,  1.2884, -1.2221,  0.0753,  0.8671, -0.5116,
         0.0343, -0.8781, -1.4735,  0.4314,  0.8192,  0.9596, -0.1029, -0.5632,
        -0.5972, -1.1962,  0.9251, -0

In [13]:
import numpy as np
from numpy.linalg import norm

def cosine_similarity(a, b): # 두개의 단어를 넣으면 코사인유사도를 알려줌
    cosine = np.dot(b, a) / (norm(b, axis = 1) * norm(a))
    return cosine

def top_n_index(cosine_matrix, n): # 유사도에서 가장 가까운걸 보여주는
    closest_indexes = cosine_matrix.argsort()[::-1]
    top_n = closest_indexes[1: n + 1]
    return top_n
cosine_matrix = cosine_similarity(token_embedding, embedding_matrix)
top_n = top_n_index(cosine_matrix, n = 5)

for index in top_n:
    print(id_to_token[index], cosine_matrix[index])

다리 0.31811273
완벽한 0.30233306
컬트 0.28272542
순수하고 0.27124792
어색한 0.2646946
